# MLflow — du suivi des expériences au Model Registry

**Cours MLOps — notebook 4/4.** Le notebook 03 a enregistré chaque modèle comme un *run*
MLflow. Ici, on exploite ces runs comme le ferait un pipeline automatisé :

1. interroger MLflow pour lister et comparer les runs ;
2. désigner le **champion** selon une règle explicite ;
3. recharger le modèle depuis MLflow et vérifier ses performances ;
4. l'enregistrer dans le **Model Registry** et le promouvoir (alias `champion`) ;
5. l'exporter vers `artifacts/mlflow/` pour l'image Docker de l'API ;
6. vérifier la chaîne complète avec le preprocessing de l'API ;
7. versionner l'export avec DVC.

> Configuration : `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME` et `MLFLOW_TRACKING_PASSWORD`
> dans `.env` (voir `README_GITHUB_DVC_MLFLOW.md`). Sans `.env`, le notebook utilise le suivi
> local créé par le notebook 03.

## 0. Configuration

In [ ]:
import json
import os
import pickle
import shutil
import sys
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)

from project_setup import (ROOT, DATA_DIR, PREPROCESSING_DIR, REGISTRY_DIR,
                           dvc_versions, git_commit, setup_mlflow)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

experiment = setup_mlflow()
client = MlflowClient()

REGISTERED_MODEL_NAME = os.getenv('MLFLOW_REGISTERED_MODEL', 'Fraud_Detection_Production')
CHAMPION_ALIAS = 'champion'
SELECTION_METRIC = 'val_pr_auc'
# Dossier copié par backend/Dockerfile : son nom est conservé pour compatibilité,
# quel que soit l'algorithme du champion (le vrai nom figure dans metadata.json)
EXPORT_DIR = REGISTRY_DIR / 'Best_Fraud_LightGBM' / '1.0.0'

## 1. Lire les runs de l'expérience

`mlflow.search_runs()` renvoie un DataFrame pandas : filtre et tri sont exécutés par le serveur.
C'est la brique de base de toute automatisation (le script Jenkins `register_best_model.py`
fait la même chose).

In [ ]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"metrics.{SELECTION_METRIC} > 0 and attributes.status = 'FINISHED'",
    order_by=[f"metrics.{SELECTION_METRIC} DESC"],
)
if runs.empty:
    raise RuntimeError(f"Aucun run avec la métrique {SELECTION_METRIC} dans '{experiment.name}' : "
                       "exécutez d'abord 03_fraud_detection_modeling.ipynb.")

columns = ['run_id', 'tags.mlflow.runName', 'params.stage', 'params.imbalance_strategy',
           'metrics.val_pr_auc', 'metrics.val_roc_auc', 'metrics.val_f1_at_best_threshold',
           'metrics.val_best_threshold', 'metrics.training_time_seconds', 'tags.selected',
           'tags.git_commit', 'start_time']
print(f"{len(runs)} runs trouvés dans l'expérience '{experiment.name}'")
runs[[c for c in columns if c in runs.columns]].head(15)

## 2. Désigner le champion

**Règle** : le run de meilleure PR-AUC de validation. La règle est écrite dans le code et
reproductible : n'importe qui peut la ré-exécuter et obtenir le même résultat.

Si l'expérience contient des runs de plusieurs exécutions du notebook 03, le champion est le
meilleur toutes exécutions confondues. On vérifie qu'il a bien été produit avec le code et
les données actuels (tags `git_commit` et `dvc_artifacts_md5`).

In [ ]:
champion = runs.iloc[0]
champion_run_id = champion['run_id']
champion_name = champion.get('params.model_name', 'inconnu')
champion_threshold = float(champion.get('metrics.val_best_threshold', 0.5))

current_versions = dvc_versions()
same_data = champion.get('tags.dvc_artifacts_md5') == current_versions.get('dvc_artifacts_md5')

print(f"🏆 Champion          : {champion['tags.mlflow.runName']} (run {champion_run_id})")
print(f"   PR-AUC validation : {champion[f'metrics.{SELECTION_METRIC}']:.4f}")
print(f"   Seuil de décision : {champion_threshold:.3f}")
print(f"   Commit Git        : {champion.get('tags.git_commit')}")
print(f"   Données DVC       : {'identiques aux données actuelles' if same_data else 'DIFFÉRENTES des données actuelles (ou non tracées)'}")

## 3. Recharger le modèle depuis MLflow et vérifier ses performances

Le modèle est téléchargé depuis l'*artifact store* (DagsHub ou `mlruns/`), puis évalué sur le
jeu de test du notebook 02. On doit retrouver les métriques `test_*` enregistrées dans le run —
c'est la garantie que l'artefact stocké est bien le modèle évalué.

In [ ]:
model = mlflow.sklearn.load_model(f"runs:/{champion_run_id}/model")
print(f"Modèle chargé : {type(model).__name__}")

with open(PREPROCESSING_DIR / 'preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)
X_test, y_test = data['X_test'], data['y_test']

proba_test = model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= champion_threshold).astype(int)
test_metrics = {
    'pr_auc': average_precision_score(y_test, proba_test),
    'roc_auc': roc_auc_score(y_test, proba_test),
    'precision': precision_score(y_test, pred_test, zero_division=0),
    'recall': recall_score(y_test, pred_test),
    'f1': f1_score(y_test, pred_test),
}

logged = {k: champion.get(f'metrics.test_{k}') for k in test_metrics}
pd.DataFrame({'recalculé': test_metrics, 'enregistré dans MLflow': logged}).round(4)

## 4. Enregistrer et promouvoir dans le Model Registry

Le **Model Registry** donne un nom stable au modèle et numérote ses versions
(`Fraud_Detection_Production` v1, v2...). Chaque version pointe vers le run qui l'a produite.

La **promotion** se fait par un *alias* : `champion` désigne la version servie en production.
Promouvoir une nouvelle version revient à déplacer l'alias ; revenir en arrière, à le
replacer sur l'ancienne. Les alias remplacent les *stages* (`Staging`, `Production`), dépréciés
depuis MLflow 2.9 ; on se rabat sur le stage `Production` si le serveur ne gère pas les alias.

In [ ]:
version = mlflow.register_model(f"runs:/{champion_run_id}/model", REGISTERED_MODEL_NAME)
print(f"Enregistré : {REGISTERED_MODEL_NAME} version {version.version}")

client.update_model_version(
    REGISTERED_MODEL_NAME, version.version,
    description=(f"{champion['tags.mlflow.runName']} — PR-AUC validation "
                 f"{champion[f'metrics.{SELECTION_METRIC}']:.4f}, seuil {champion_threshold:.3f}"),
)
version_tags = {
    'algorithm': champion_name,
    'threshold': f"{champion_threshold:.6f}",
    'selection_metric': SELECTION_METRIC,
    'test_pr_auc': f"{test_metrics['pr_auc']:.4f}",
    'git_commit': champion.get('tags.git_commit') or 'inconnu',
    **{k: str(v) for k, v in current_versions.items()},
}
for key, value in version_tags.items():
    client.set_model_version_tag(REGISTERED_MODEL_NAME, version.version, key, value)

try:
    client.set_registered_model_alias(REGISTERED_MODEL_NAME, CHAMPION_ALIAS, version.version)
    production_uri = f"models:/{REGISTERED_MODEL_NAME}@{CHAMPION_ALIAS}"
except Exception as exc:   # serveur sans prise en charge des alias
    print(f"Alias indisponible ({exc}) : utilisation du stage Production.")
    client.transition_model_version_stage(REGISTERED_MODEL_NAME, version.version,
                                          stage='Production', archive_existing_versions=True)
    production_uri = f"models:/{REGISTERED_MODEL_NAME}/Production"
print(f"Version de production : {production_uri}")

In [ ]:
history = []
for v in client.search_model_versions(f"name = '{REGISTERED_MODEL_NAME}'"):
    details = client.get_model_version(REGISTERED_MODEL_NAME, v.version)   # inclut les alias
    history.append({
        'version': int(v.version), 'run_id': v.run_id, 'statut': v.status,
        'alias': ', '.join(getattr(details, 'aliases', None) or []), 'stage': v.current_stage,
        'créée le': datetime.fromtimestamp(v.creation_timestamp / 1000).strftime('%Y-%m-%d %H:%M'),
    })
pd.DataFrame(history).sort_values('version', ascending=False)

## 5. Charger le modèle de production par son nom

Le code consommateur ne connaît ni le run ni la version : seulement le nom et l'alias.

In [ ]:
production_model = mlflow.sklearn.load_model(production_uri)
same_predictions = np.allclose(production_model.predict_proba(X_test.head(1_000))[:, 1], proba_test[:1_000])
print(f"Chargé depuis {production_uri} — prédictions identiques au champion : {same_predictions}")
assert same_predictions

## 6. Exporter le modèle pour l'API

L'image Docker du backend ne contacte pas MLflow au démarrage : `backend/Dockerfile` copie
`artifacts/mlflow/Best_Fraud_LightGBM/1.0.0/` (fichiers `model.pkl` et `metadata.json`), et
`test/test_fraud_scenario.py` lit `artifacts/mlflow/Best_Fraud_LightGBM/production.pkl`. On écrit
ces trois fichiers, au même format que le script Jenkins `register_best_model.py`.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
with open(EXPORT_DIR / 'model.pkl', 'wb') as f:
    pickle.dump(production_model, f)
shutil.copy2(EXPORT_DIR / 'model.pkl', EXPORT_DIR.parent / 'production.pkl')

export_metadata = {
    'model_name': f"Best_Fraud_{champion_name}",
    'version': '1.0.0',
    'run_id': champion_run_id,
    'algorithm': champion_name,
    'roc_auc': float(test_metrics['roc_auc']),
    'pr_auc': float(test_metrics['pr_auc']),
    'threshold': champion_threshold,
    'registered_model': REGISTERED_MODEL_NAME,
    'registered_version': int(version.version),
    'selection_metric': SELECTION_METRIC,
    'exported_at': datetime.now().isoformat(timespec='seconds'),
    'git_commit': git_commit(),
    **current_versions,
}
with open(EXPORT_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(export_metadata, f, indent=2, ensure_ascii=False)

for path in [EXPORT_DIR / 'model.pkl', EXPORT_DIR / 'metadata.json', EXPORT_DIR.parent / 'production.pkl']:
    print(f"✓ {path.relative_to(ROOT)}")

## 7. Vérification de bout en bout avec le preprocessing de l'API

On rejoue le chemin d'une requête `/predict` : transactions **brutes** → `InferencePreprocessor`
(code de l'API) → modèle exporté. Il s'agit d'un test d'intégration, pas d'une évaluation (une
partie de ces transactions a servi à l'entraînement). La cellule nécessite les dépendances du
backend ; sinon, elle est ignorée.

In [ ]:
sys.path.insert(0, str(ROOT / 'backend' / 'src'))
try:
    from api import InferencePreprocessor
except ImportError as exc:
    print(f"Vérification ignorée : dépendance du backend manquante ({exc}).")
else:
    raw = pd.read_csv(DATA_DIR / 'fraud_600k.csv')
    frauds = raw[raw['is_fraud'] == 1]
    sample = pd.concat([frauds.sample(min(200, len(frauds)), random_state=0),
                        raw[raw['is_fraud'] == 0].sample(800, random_state=0)])

    with open(EXPORT_DIR / 'model.pkl', 'rb') as f:
        exported_model = pickle.load(f)
    X_api = InferencePreprocessor().preprocess_inference(sample.drop(columns=['is_fraud']))
    proba_api = exported_model.predict_proba(X_api)[:, 1]

    print(f"\nFraudes détectées : {int(((proba_api >= champion_threshold) & (sample['is_fraud'] == 1)).sum())}"
          f" / {int(sample['is_fraud'].sum())} (seuil {champion_threshold:.3f})")
    print(f"Fausses alertes   : {int(((proba_api >= champion_threshold) & (sample['is_fraud'] == 0)).sum())}"
          f" / {int((sample['is_fraud'] == 0).sum())}")

## 8. Versionner le modèle exporté

Le modèle est désormais tracé à trois endroits complémentaires :

| Où | Quoi | Pour qui |
|---|---|---|
| MLflow (runs) | paramètres, métriques, artefacts de chaque essai | data scientists |
| MLflow Model Registry | versions nommées, alias `champion` | décision de mise en production |
| DVC (`artifacts.dvc`) | fichiers exacts embarqués dans l'image Docker | pipeline de déploiement |

Depuis la racine du dépôt :

```bash
dvc add artifacts
git add artifacts.dvc
git commit -m "model: export champion <nom> v<version>"
dvc push          # avant git push : sinon les autres ne peuvent pas récupérer les fichiers
git push
```

Le pipeline Jenkins de déploiement (`dvc pull` puis `docker build`) embarquera alors ce modèle.

In [ ]:
print("RÉSUMÉ")
print("=" * 70)
print(f"Expérience       : {experiment.name} ({len(runs)} runs)")
print(f"Champion         : {champion['tags.mlflow.runName']} — run {champion_run_id}")
print(f"Model Registry   : {REGISTERED_MODEL_NAME} v{version.version} ({production_uri})")
print(f"Test             : PR-AUC {test_metrics['pr_auc']:.4f} | ROC-AUC {test_metrics['roc_auc']:.4f} | "
      f"F1 {test_metrics['f1']:.4f} (seuil {champion_threshold:.3f})")
print(f"Export API       : {EXPORT_DIR.relative_to(ROOT)}")